# Dat200 CA5

# Imports

# Reading data

# Data exploration

# Data visualization

# Data cleaning

# Pipeline A

In [ ]:
## Preprocessing

In [ ]:
def replace_invalid_values(df: pd.DataFrame, valid_ranges: dict) -> pd.DataFrame:
    """Replace values outside the specified valid ranges with NaN."""
    df = df.copy()

    for column, (lower, upper) in valid_ranges.items():

        if lower is not None:
            df.loc[df[column] < lower, column] = np.nan

        if upper is not None:
            df.loc[df[column] > upper, column] = np.nan

    return df


X_train = replace_invalid_values(X_train, valid_ranges)
X_test = replace_invalid_values(X_test, valid_ranges)

In [ ]:
# Identify numerical and categorical features
num_cols = X_train.select_dtypes(include='number').columns.tolist()
cat_cols = X_train.select_dtypes(exclude='number').columns.tolist()

## Pipelines

In [ ]:
# Numerical Pipeline
numeric_pipeline = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler', StandardScaler())
])

# Categorical Pipeline
categorical_pipeline = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('onehot', OneHotEncoder(handle_unknown='ignore'))
])

# Combining the pipelines with ColumnTransformer
preprocessor = ColumnTransformer(transformers=[
    ('num', numeric_pipeline, num_cols),
    ('cat', categorical_pipeline, cat_cols)
])

# Numerical pipeline for tree-based models
numeric_pipeline_tree = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='median'))
])

# Preprocessor for Random Forest
preprocessor_rf = ColumnTransformer(transformers=[
    ('num', numeric_pipeline_tree, num_cols),
    ('cat', categorical_pipeline, cat_cols)
])

## Evaluation function

In [ ]:
def evaluate_regression(
    model,
    X_train,
    X_test,
    y_train,
    y_test
):
    """Evaluate a regression model on training and test data."""

    # Generate predictions
    y_train_pred = np.asarray(model.predict(X_train)).ravel()
    y_test_pred = np.asarray(model.predict(X_test)).ravel()

    # Calculate test metrics
    mae = mean_absolute_error(y_test, y_test_pred)
    mse = mean_squared_error(y_test, y_test_pred)
    r2 = r2_score(y_test, y_test_pred)

    # Compare training and test MAE
    mae_train = mean_absolute_error(y_train, y_train_pred)
    mae_test = mean_absolute_error(y_test, y_test_pred)
    mae_difference = mae_test - mae_train

    return {
        'MAE': mae,
        'MSE': mse,
        'R²': r2,
        'Training MAE': mae_train,
        'Test MAE': mae_test,
        'MAE difference': mae_difference
    }

## Polynomial Regression + Ridge

In [ ]:
# Define the pipeline
poly_ridge_pipeline = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('poly', PolynomialFeatures(
        degree=3,
        include_bias=False
    )),
    ('poly_scaler', StandardScaler()),
    ('model', Ridge())
])

In [ ]:
# Define cross-validation strategy
cv = KFold(
    n_splits=5,
    shuffle=True,
    random_state=42
)

In [ ]:
# Define hyperparameter search space
poly_ridge_param_grid = {
    'model__alpha': [0.001, 0.01, 0.1, 1, 10, 100, 500]
}

# Perform hyperparameter optimization
poly_ridge_search = GridSearchCV(
    estimator=poly_ridge_pipeline,
    param_grid=poly_ridge_param_grid,
    scoring='neg_mean_absolute_error',
    cv=cv,
    n_jobs=1,
    verbose=2
)

poly_ridge_search.fit(X_train, y_train)

In [ ]:
# Display the best hyperparameters and CV score
print('Best parameters:', poly_ridge_search.best_params_)
print('Best CV MAE:', -poly_ridge_search.best_score_)

In [ ]:
# Retrieve the best model
best_poly_ridge = poly_ridge_search.best_estimator_

In [ ]:
# Evaluate the best model
poly_ridge_results = evaluate_regression(
    best_poly_ridge,
    X_train,
    X_test,
    y_train,
    y_test
)

poly_ridge_results

## Polynomial Regression

In [ ]:
# Define the pipeline
poly_pipeline = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('poly', PolynomialFeatures(
        degree=3,
        include_bias=False
    )),
    ('model', LinearRegression())
])

In [ ]:
# Train the model
poly_pipeline.fit(X_train, y_train)

In [ ]:
# Evaluate the model
poly_results = evaluate_regression(
    poly_pipeline,
    X_train,
    X_test,
    y_train,
    y_test
)

poly_results

# Pipeline B

# Pipeline C

# Model comparison and evaluation

# Kaggle submission